In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import pickle
import os
from torch_geometric.datasets import BA2MotifDataset
from torch.utils.data import Subset
from torch_geometric.data import Data
import networkx as ntx
from torch_geometric.utils import to_networkx,subgraph
import matplotlib.pyplot as plt

c:\Users\mahboub\miniforge3\envs\GNN\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\mahboub\miniforge3\envs\GNN\Lib\site-packages\torch_scatter\_version_cpu.pyd
  import torch_geometric.typing
c:\Users\mahboub\miniforge3\envs\GNN\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\mahboub\miniforge3\envs\GNN\Lib\site-packages\torch_cluster\_version_cpu.pyd
  import torch_geometric.typing
c:\Users\mahboub\miniforge3\envs\GNN\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\mahboub\miniforge3\envs\GNN\Lib\site-packages\torch_spline_conv\_version_cpu.pyd
  import torch_geometric.typing
c:

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [4]:
dataset_path='./data/BA2Motif'
with open('./data/splits.pkl', 'rb') as f:
    splits=pickle.load(f)

full_dataset=BA2MotifDataset(root=dataset_path)
test_dataset=Subset(full_dataset,splits['test'])

with open('./data/test_result.pkl', 'rb') as f:
    test_result=pickle.load(f)


In [5]:
class GINGraphClf(nn.Module):
    def __init__(self, in_dim,out_dim, hidden_dim=64):
        super().__init__()
        self.conv1=GINConv(nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv2=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv3=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        
        self.mlp = nn.Sequential(
            nn.Linear(3*hidden_dim, hidden_dim//2),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, out_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x, edge_index, batch):
        h1=self.conv1(x, edge_index)
        h2=self.conv2(h1, edge_index)
        h3=self.conv3(h2, edge_index)

        h1=global_add_pool(h1,batch)
        h2=global_add_pool(h2,batch)
        h3=global_add_pool(h3,batch)    


        x=torch.cat([h1,h2,h3],dim=1)


        return self.mlp(x)

In [6]:
model=GINGraphClf(test_dataset[0].num_features,2).to(device)


checkpoint=torch.load('./models/GIN/best_gin_model.pt',map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [7]:
def compute_node_saliency(model, data, target_class, device):

    model.eval()
    data = data.to(device)
    data.x.requires_grad_(True)
    
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    out = model(data.x, data.edge_index, batch)  # [1, num_classes]
    score = out[0, target_class]  
    
    score.backward()
    grad = data.x.grad  # [num_nodes, num_features]
    
    saliency = torch.norm(grad, dim=1)  # [num_nodes]

    data.x.grad = None
    data.x.requires_grad_(False)
    
    return saliency.cpu()

In [8]:
def induced_subgraph(data, node_indices):
    
    node_set = torch.tensor(node_indices, dtype=torch.long)
    edge_index, _ = subgraph(node_set, data.edge_index, relabel_nodes=True)
    x = data.x[node_set]
    sub_data = Data(x=x, edge_index=edge_index)
    if hasattr(data, 'y'):
        sub_data.y = data.y
    return sub_data

In [9]:
def ensure_connectivity(sub_data, original_data, target_size):
    
    G_sub = to_networkx(sub_data, to_undirected=True)
    if ntx.is_connected(G_sub):
        return sub_data, list(G_sub.nodes)
    
    components = list(ntx.connected_components(G_sub))

    largest_comp = max(components, key=len)
    current_nodes = set(largest_comp)
    
    final_nodes = sorted(list(current_nodes))
    connected_data = induced_subgraph(original_data, final_nodes)
    return connected_data, final_nodes

In [10]:
def compute_score(model,data,target_class,device):
    model.eval()
    with torch.no_grad():
        if data.batch is not None:
            out=model(data.x,data.edge_index,data.batch)
        else:
            batch=torch.zeros(data.num_nodes,dtype=torch.long,device=device)
            out = model(data.x, data.edge_index, batch)
        if out.dim() == 1:
            out = out.unsqueeze(0)
        score = out[0, target_class]  
    return score 



In [11]:
def ensure_connected_with_shortest_paths(original_data, selected_nodes, target_size):
    
    G_full = to_networkx(original_data, to_undirected=True)
    
    sub_G = G_full.subgraph(selected_nodes).copy()
    
    if ntx.is_connected(sub_G):
        return selected_nodes, induced_subgraph(original_data, selected_nodes)
    
    components = list(ntx.connected_components(sub_G))
    components.sort(key=len, reverse=True)
    
    final_nodes = set(components[0])
    
    for comp in components[1:]:
        min_path = None
        min_path_len = float('inf')
        for u in comp:
            for v in final_nodes:
                try:
                    path = ntx.shortest_path(G_full, source=u, target=v)
                    if len(path) < min_path_len:
                        min_path_len = len(path)
                        min_path = path
                except ntx.NetworkXNoPath:
                    continue
        if min_path is not None:
            final_nodes.update(min_path) 
        else:
            final_nodes.update(comp)
    
    final_nodes = sorted(list(final_nodes))
    final_subgraph = induced_subgraph(original_data, final_nodes)
    return final_nodes, final_subgraph

In [12]:
def baseline_explanation(model, data, target_class, budget, device):

    saliency = compute_node_saliency(model, data, target_class, device)
    num_nodes = data.num_nodes
    if isinstance(budget, float):
        k = max(1, int(num_nodes * budget))
    else:
        k = min(budget, num_nodes)
    
    _, top_indices = torch.topk(saliency, k)
    selected_nodes = top_indices.tolist()
    
    data_cpu = data.cpu()
    sub_data_initial = induced_subgraph(data_cpu, selected_nodes)
    
    final_nodes, explanation_graph = ensure_connected_with_shortest_paths(
        data_cpu, selected_nodes, k
    )
    
    original_data_score=compute_score(model,data,target_class,device=device)
    subgraph_score=compute_score(model,explanation_graph,target_class,device=device)

    fidelity_score=original_data_score-subgraph_score
    
    return explanation_graph, final_nodes,round(fidelity_score.item(),4)

In [13]:
def visualize_explanation(original_data, explanation_node_indices, title="Explanation Subgraph", save_path=None):

    G = to_networkx(original_data, to_undirected=True)
    pos = ntx.spring_layout(G, seed=404131029)  
    
    node_colors = []
    for node in G.nodes():
        if node in explanation_node_indices:
            node_colors.append('red')
        else:
            node_colors.append('skyblue')
    
    plt.figure(figsize=(10, 8))
    ntx.draw(G, pos, node_color=node_colors, with_labels=True, 
            node_size=500, font_size=10, font_weight='bold', 
            edge_color='gray', alpha=0.8)
    
    plt.title(f"{title}\n(Explanation nodes: {len(explanation_node_indices)} / {original_data.num_nodes})")
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    


In [14]:
fidelity_dorp_score=0
for key in test_result.keys():
    if len(test_result[key]) == 0:
        continue
    max_pic=10
    if len(test_result[key])<max_pic:
        max_pic=len(test_result[key])
    
    random_indices=torch.randint(0,len(test_result[key])-1,(max_pic,)).tolist()
    for i in range(len(test_result[key])):
        data=test_dataset[test_result[key][i]['index']]
        pred_class=test_result[key][i]['pred']
        explanation_graph, selected_nodes,fidelity_score = baseline_explanation(
            model, data, target_class=pred_class, budget=0.2, device=device
        )
        fidelity_dorp_score+=fidelity_score
        if i in random_indices:
            visualize_explanation(data.cpu(), selected_nodes,save_path=f"./explanation/baseline/{key}/index_{test_result[key][i]['index']}.png")  

print(f" fidelity score: {fidelity_dorp_score/len(test_dataset)}")


 fidelity score: 0.521706
